# Lesson 4: GLiNER - Zero-Shot Named Entity Recognition

## 🎯 Learning Objectives

By the end of this lesson, you will:
1. Understand what zero-shot NER is and why it matters
2. Master the GLiNER architecture and how it works
3. Use GLiNER for custom entity extraction without training
4. Compare GLiNER with traditional NER approaches
5. Learn best practices for prompt engineering with GLiNER

---

## 📚 Table of Contents

1. [Introduction to Zero-Shot NER](#1-introduction-to-zero-shot-ner)
2. [GLiNER Architecture Deep Dive](#2-gliner-architecture-deep-dive)
3. [Getting Started with GLiNER](#3-getting-started-with-gliner)
4. [Custom Entity Types](#4-custom-entity-types)
5. [Advanced Usage & Configuration](#5-advanced-usage--configuration)
6. [Comparing GLiNER vs Traditional NER](#6-comparing-gliner-vs-traditional-ner)
7. [Best Practices & Tips](#7-best-practices--tips)
8. [Further Reading](#8-further-reading)

---

## 📦 Setup & Installation

In [ ]:
# Install required packages
!pip install -q gliner torch transformers

In [ ]:
# Import libraries
import torch
from gliner import GLiNER
import time
import warnings
warnings.filterwarnings('ignore')

# Check device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")

---

## 1. Introduction to Zero-Shot NER

### The Problem with Traditional NER

Traditional NER models have a fundamental limitation:

```
Traditional NER:
┌──────────────────┐    ┌─────────────────┐    ┌─────────────────┐
│  Training Data   │ -> │  Train Model    │ -> │  Fixed Labels   │
│  (PER, ORG, LOC) │    │  (weeks/months) │    │  (PER, ORG, LOC)│
└──────────────────┘    └─────────────────┘    └─────────────────┘
                                                        │
                              Need "PRODUCT"? ─────────┘
                              Start over! 😫
```

### The Zero-Shot Solution

Zero-shot NER allows extraction of **any entity type** without retraining:

```
Zero-Shot NER (GLiNER):
┌──────────────────┐    ┌─────────────────┐
│  Text + Labels   │ -> │   GLiNER Model  │ -> Entities!
│  (any labels!)   │    │   (pre-trained) │
└──────────────────┘    └─────────────────┘

Labels: ["person", "company", "product", "skill", "anything!"] ✅
```

### Why Zero-Shot NER Matters

| Challenge | Traditional NER | Zero-Shot NER |
|-----------|----------------|---------------|
| New entity types | Collect data, retrain | Just add label |
| Domain adaptation | Fine-tuning required | Works out of box |
| Prototype speed | Days to weeks | Minutes |
| Annotation cost | High | Zero |
| Flexibility | Limited | Unlimited |

---

## 2. GLiNER Architecture Deep Dive

### What is GLiNER?

> **GLiNER** (Generalist and Lightweight model for Named Entity Recognition) is a zero-shot NER model that can identify any entity type using bidirectional transformer encoders. It was published at NAACL 2024.
>
> — [Zaratiana et al., 2023](https://arxiv.org/abs/2311.08526)

### Architecture Overview

```
                    Input
                      │
         ┌────────────┴────────────┐
         │                         │
    Entity Types              Text Tokens
[ENT]person[ENT]org...    "Barack Obama visited..."
         │                         │
         └────────────┬────────────┘
                      │
              ┌───────▼───────┐
              │   BERT-like   │
              │   Encoder     │
              └───────┬───────┘
                      │
         ┌────────────┴────────────┐
         │                         │
  Entity Embeddings         Span Embeddings
  (from [ENT] tokens)       (from text tokens)
         │                         │
    ┌────▼────┐              ┌─────▼─────┐
    │   FFN   │              │   Span    │
    │         │              │   Layer   │
    └────┬────┘              └─────┬─────┘
         │                         │
         └─────────┬───────────────┘
                   │
            ┌──────▼──────┐
            │  Matching   │
            │ (dot prod + │
            │  sigmoid)   │
            └──────┬──────┘
                   │
              Predictions
```

### Key Innovations

1. **Entity Type Prompts**: Entity types are encoded alongside text
2. **Span Representations**: All possible spans are considered
3. **Matching Score**: Entity types are matched to spans via dot product
4. **Efficiency**: ~10x smaller than LLMs with comparable performance

### Mathematical Formulation

For a span $(i, j)$ and entity type $e$:

$$\text{score}(i, j, e) = \sigma(\mathbf{s}_{i,j}^T \cdot \mathbf{e})$$

Where:
- $\mathbf{s}_{i,j}$ is the span representation
- $\mathbf{e}$ is the entity type embedding
- $\sigma$ is the sigmoid function

---

## 3. Getting Started with GLiNER

### Loading the Model

In [ ]:
# Load GLiNER model
model = GLiNER.from_pretrained("urchade/gliner_medium-v2.1")

print("✅ GLiNER model loaded successfully!")
print(f"   Model: urchade/gliner_medium-v2.1")

In [ ]:
# Basic usage
text = """
Apple Inc. was founded by Steve Jobs, Steve Wozniak, and Ronald Wayne 
in Cupertino, California on April 1, 1976. The company is known for 
products like the iPhone, MacBook, and Apple Watch.
"""

# Define entity types (the magic of zero-shot!)
labels = ["person", "company", "location", "product", "date"]

# Extract entities
entities = model.predict_entities(text, labels, threshold=0.5)

print("🏷️ Extracted Entities:\n")
print(f"{'Entity':<30} {'Label':<12} {'Score'}")
print("=" * 55)

for entity in entities:
    print(f"{entity['text']:<30} {entity['label']:<12} {entity['score']:.4f}")

In [ ]:
# Understanding the output format
print("📋 Entity Structure:\n")

for entity in entities[:3]:
    print(f"Entity: '{entity['text']}'")
    print(f"   Label: {entity['label']}")
    print(f"   Score: {entity['score']:.4f}")
    print(f"   Start: {entity['start']}")
    print(f"   End: {entity['end']}")
    print()

### Available Models

| Model | Size | Speed | Accuracy |
|-------|------|-------|----------|
| `urchade/gliner_small-v2.1` | ~50M | Fastest | Good |
| `urchade/gliner_medium-v2.1` | ~110M | Balanced | Better |
| `urchade/gliner_large-v2.1` | ~340M | Slower | Best |
| `urchade/gliner_multi-v2.1` | ~110M | Balanced | Multilingual |

> **Reference**: [GLiNER GitHub](https://github.com/urchade/GLiNER)

---

## 4. Custom Entity Types

### The Power of Custom Labels

GLiNER can extract **any** entity type you specify:

In [ ]:
# Example 1: Medical Domain
medical_text = """
The patient was diagnosed with Type 2 Diabetes and prescribed Metformin 500mg 
twice daily. They also have a history of hypertension, treated with Lisinopril.
Follow-up appointment scheduled with Dr. Smith at Mayo Clinic.
"""

medical_labels = ["disease", "medication", "dosage", "doctor", "hospital"]

medical_entities = model.predict_entities(medical_text, medical_labels, threshold=0.4)

print("🏥 Medical NER:\n")
print(f"Text: {medical_text.strip()}\n")

for entity in medical_entities:
    print(f"   {entity['text']:<20} → {entity['label']} ({entity['score']:.3f})")

In [ ]:
# Example 2: Legal Domain
legal_text = """
In the case of Smith v. Johnson (2023), the Supreme Court ruled that 
Section 230 of the Communications Decency Act does not provide immunity 
for AI-generated content. Justice Roberts delivered the majority opinion.
"""

legal_labels = ["case name", "court", "law", "judge", "year"]

legal_entities = model.predict_entities(legal_text, legal_labels, threshold=0.4)

print("⚖️ Legal NER:\n")
print(f"Text: {legal_text.strip()}\n")

for entity in legal_entities:
    print(f"   {entity['text']:<40} → {entity['label']} ({entity['score']:.3f})")

In [ ]:
# Example 3: E-commerce/Product Domain
ecommerce_text = """
The Sony WH-1000XM5 wireless headphones are priced at $349.99 and come 
in black and silver colors. They feature 30-hour battery life and 
industry-leading noise cancellation. Free shipping on orders over $50.
"""

ecommerce_labels = ["product name", "brand", "price", "color", "feature"]

ecommerce_entities = model.predict_entities(ecommerce_text, ecommerce_labels, threshold=0.4)

print("🛒 E-commerce NER:\n")
print(f"Text: {ecommerce_text.strip()}\n")

for entity in ecommerce_entities:
    print(f"   {entity['text']:<30} → {entity['label']} ({entity['score']:.3f})")

In [ ]:
# Example 4: Resume/Job Domain
resume_text = """
John Smith is a Senior Software Engineer with 8 years of experience. 
He holds a Master's degree in Computer Science from MIT. His skills 
include Python, TensorFlow, and Kubernetes. Previously worked at Google 
and Amazon. Fluent in English and Spanish.
"""

resume_labels = ["person", "job title", "skill", "company", "degree", "university", "language"]

resume_entities = model.predict_entities(resume_text, resume_labels, threshold=0.4)

print("📄 Resume NER:\n")
print(f"Text: {resume_text.strip()}\n")

for entity in resume_entities:
    print(f"   {entity['text']:<25} → {entity['label']} ({entity['score']:.3f})")

In [ ]:
# Example 5: Scientific/Research Domain
science_text = """
The CRISPR-Cas9 gene editing technique was developed by Jennifer Doudna 
and Emmanuelle Charpentier. Their 2012 paper in Science demonstrated 
how the system could be used for precise DNA modification. They received 
the Nobel Prize in Chemistry in 2020.
"""

science_labels = ["scientist", "technique", "publication", "award", "year", "field"]

science_entities = model.predict_entities(science_text, science_labels, threshold=0.4)

print("🔬 Scientific NER:\n")
print(f"Text: {science_text.strip()}\n")

for entity in science_entities:
    print(f"   {entity['text']:<35} → {entity['label']} ({entity['score']:.3f})")

---

## 5. Advanced Usage & Configuration

### Threshold Tuning

In [ ]:
# Threshold affects precision/recall trade-off
text = "Microsoft CEO Satya Nadella announced new AI features in Seattle."
labels = ["person", "company", "city", "technology"]

print("🎚️ Threshold Comparison:\n")
print(f"Text: {text}\n")

for threshold in [0.3, 0.5, 0.7, 0.9]:
    entities = model.predict_entities(text, labels, threshold=threshold)
    entity_list = [(e['text'], e['label'], f"{e['score']:.2f}") for e in entities]
    print(f"Threshold {threshold}: {entity_list}")

In [ ]:
# Flat NER vs Nested NER
text = "The New York Times reported on United Nations headquarters in New York."
labels = ["organization", "city", "newspaper"]

print("📰 Handling Nested Entities:\n")
print(f"Text: {text}\n")

# Default (flat) - may miss nested entities
flat_entities = model.predict_entities(text, labels, threshold=0.4, flat_ner=True)
print("Flat NER (default):")
for e in flat_entities:
    print(f"   {e['text']:<30} → {e['label']}")

# Nested - can detect overlapping entities
nested_entities = model.predict_entities(text, labels, threshold=0.4, flat_ner=False)
print("\nNested NER:")
for e in nested_entities:
    print(f"   {e['text']:<30} → {e['label']}")

In [ ]:
# Batch processing
texts = [
    "Google acquired YouTube for $1.65 billion in 2006.",
    "Tesla CEO Elon Musk tweeted about Bitcoin.",
    "The iPhone 15 Pro features a titanium design."
]

labels = ["company", "person", "product", "amount", "year"]

print("⚡ Batch Processing:\n")

# Process multiple texts
for i, text in enumerate(texts):
    entities = model.predict_entities(text, labels, threshold=0.5)
    print(f"Text {i+1}: {text}")
    print(f"Entities: {[(e['text'], e['label']) for e in entities]}\n")

### Label Formatting Best Practices

In [ ]:
# GLiNER works best with lowercase or title case labels
text = "Barack Obama served as the 44th President of the United States from 2009 to 2017."

# Different label formats
label_formats = [
    ["person", "title", "country", "year"],           # lowercase (recommended)
    ["Person", "Title", "Country", "Year"],           # title case (also good)
    ["PERSON", "TITLE", "COUNTRY", "YEAR"],           # uppercase (less reliable)
    ["person name", "job title", "country name", "year"],  # descriptive (can help)
]

print("🏷️ Label Format Comparison:\n")
print(f"Text: {text}\n")

for labels in label_formats:
    entities = model.predict_entities(text, labels, threshold=0.4)
    entity_list = [(e['text'], e['label']) for e in entities]
    print(f"Labels {labels}:")
    print(f"   Found: {entity_list}\n")

---

## 6. Comparing GLiNER vs Traditional NER

### Performance Comparison

In [ ]:
# Compare GLiNER with BERT NER
from transformers import pipeline

# Load BERT NER
bert_ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple")

test_text = "Tim Cook announced that Apple will open a new office in Austin, Texas."

print("📊 GLiNER vs BERT NER Comparison\n")
print(f"Text: {test_text}\n")

# BERT NER (fixed labels)
print("BERT NER (fixed labels: PER, ORG, LOC, MISC):")
bert_entities = bert_ner(test_text)
for e in bert_entities:
    print(f"   {e['word']:<20} → {e['entity_group']} ({e['score']:.3f})")

# GLiNER with same labels
print("\nGLiNER (same labels):")
gliner_entities = model.predict_entities(test_text, ["person", "organization", "location"], threshold=0.5)
for e in gliner_entities:
    print(f"   {e['text']:<20} → {e['label']} ({e['score']:.3f})")

# GLiNER with custom labels (flexibility!)
print("\nGLiNER (custom labels):")
custom_entities = model.predict_entities(test_text, ["ceo", "tech company", "city", "state"], threshold=0.5)
for e in custom_entities:
    print(f"   {e['text']:<20} → {e['label']} ({e['score']:.3f})")

In [ ]:
# Speed comparison
import time

test_texts = [
    "Google was founded by Larry Page and Sergey Brin at Stanford University.",
] * 10  # Repeat for timing

# BERT NER timing
start = time.time()
for text in test_texts:
    _ = bert_ner(text)
bert_time = time.time() - start

# GLiNER timing
labels = ["person", "company", "university"]
start = time.time()
for text in test_texts:
    _ = model.predict_entities(text, labels, threshold=0.5)
gliner_time = time.time() - start

print("⏱️ Speed Comparison (10 texts):\n")
print(f"   BERT NER:   {bert_time:.3f}s ({bert_time/10*1000:.1f}ms per text)")
print(f"   GLiNER:     {gliner_time:.3f}s ({gliner_time/10*1000:.1f}ms per text)")

### When to Use Which?

| Scenario | Recommendation |
|----------|---------------|
| Fixed entity types (PER, ORG, LOC) | BERT NER (faster, more accurate) |
| Custom/domain-specific entities | GLiNER |
| Rapid prototyping | GLiNER |
| Production with specific schema | Fine-tuned BERT |
| No training data available | GLiNER |
| Frequently changing entity types | GLiNER |

---

## 7. Best Practices & Tips

### Label Engineering

In [ ]:
# Tip 1: Be specific with labels
text = "Dr. Jane Smith works at Massachusetts General Hospital."

# Generic labels
generic = model.predict_entities(text, ["person", "organization"], threshold=0.4)
print("Generic labels (person, organization):")
for e in generic:
    print(f"   {e['text']:<40} → {e['label']}")

# Specific labels
specific = model.predict_entities(text, ["doctor name", "hospital"], threshold=0.4)
print("\nSpecific labels (doctor name, hospital):")
for e in specific:
    print(f"   {e['text']:<40} → {e['label']}")

In [ ]:
# Tip 2: Limit number of labels for better accuracy
text = "Tesla stock (TSLA) rose 5% after Elon Musk announced record deliveries."

# Too many labels (may reduce accuracy)
many_labels = ["company", "stock ticker", "percentage", "person", "ceo", 
               "number", "metric", "announcement", "increase", "event"]
result_many = model.predict_entities(text, many_labels, threshold=0.4)
print(f"Many labels ({len(many_labels)}): {[(e['text'], e['label']) for e in result_many]}")

# Focused labels (better accuracy)
few_labels = ["company", "stock ticker", "person", "percentage"]
result_few = model.predict_entities(text, few_labels, threshold=0.4)
print(f"Few labels ({len(few_labels)}): {[(e['text'], e['label']) for e in result_few]}")

In [ ]:
# Tip 3: Use descriptive labels when needed
text = "The meeting is scheduled for 3:30 PM on December 15, 2024."

# Simple labels
simple = model.predict_entities(text, ["time", "date"], threshold=0.4)
print("Simple labels:")
for e in simple:
    print(f"   {e['text']:<20} → {e['label']}")

# Descriptive labels
descriptive = model.predict_entities(text, 
    ["time of day", "calendar date", "meeting time"], threshold=0.4)
print("\nDescriptive labels:")
for e in descriptive:
    print(f"   {e['text']:<20} → {e['label']}")

In [ ]:
# Tip 4: Handle ambiguity with multiple extractions
text = "Apple released the new Apple Watch."

# First pass: organizations
orgs = model.predict_entities(text, ["company"], threshold=0.5)
print("Companies:", [(e['text'], e['score']) for e in orgs])

# Second pass: products
products = model.predict_entities(text, ["product"], threshold=0.5)
print("Products:", [(e['text'], e['score']) for e in products])

# Combined extraction
combined = model.predict_entities(text, ["company", "product"], threshold=0.5)
print("Combined:", [(e['text'], e['label']) for e in combined])

### Production Tips

In [ ]:
# Production-ready extraction function
def extract_entities_robust(text, labels, model, threshold=0.5, max_length=512):
    """
    Robust entity extraction with error handling and validation.
    
    Args:
        text: Input text
        labels: List of entity types to extract
        model: GLiNER model instance
        threshold: Confidence threshold (0-1)
        max_length: Maximum text length to process
    
    Returns:
        List of entity dictionaries
    """
    # Input validation
    if not text or not text.strip():
        return []
    
    if not labels:
        raise ValueError("Labels list cannot be empty")
    
    # Truncate long text
    if len(text) > max_length:
        text = text[:max_length]
    
    # Normalize labels (lowercase)
    labels = [l.lower().strip() for l in labels]
    
    # Extract entities
    try:
        entities = model.predict_entities(text, labels, threshold=threshold)
    except Exception as e:
        print(f"Warning: Extraction failed - {e}")
        return []
    
    # Post-process: remove duplicates and sort by position
    seen = set()
    unique_entities = []
    
    for entity in sorted(entities, key=lambda x: x['start']):
        key = (entity['text'], entity['label'])
        if key not in seen:
            seen.add(key)
            unique_entities.append(entity)
    
    return unique_entities

# Test the robust function
text = "Apple Inc. announced new products. Apple is headquartered in Cupertino."
entities = extract_entities_robust(text, ["company", "location"], model)

print("🔧 Robust Extraction Result:")
for e in entities:
    print(f"   {e['text']:<20} → {e['label']}")

---

## 8. Further Reading

### 📚 Essential Papers

1. **GLiNER: Generalist Model for Named Entity Recognition** (2023)
   - Zaratiana, U., Tomeh, N., Holat, P., & Charnois, T.
   - [arXiv:2311.08526](https://arxiv.org/abs/2311.08526)
   - [NAACL 2024 Paper](https://aclanthology.org/2024.naacl-long.300.pdf)

2. **NuNER: Entity Recognition Encoder Pre-training via LLM-Annotated Data** (2024)
   - NuMind AI
   - [arXiv:2402.15343](https://arxiv.org/abs/2402.15343)

### 🔗 Official Resources

- [GLiNER GitHub Repository](https://github.com/urchade/GLiNER)
- [GLiNER on Hugging Face](https://huggingface.co/urchade)
- [GLiNER Demo Space](https://huggingface.co/spaces/tomaarsen/gliner_medium-v2.1)

### 📊 Benchmarks

GLiNER achieves competitive performance on zero-shot NER benchmarks:

| Benchmark | GLiNER-medium | ChatGPT | UniNER-13B |
|-----------|---------------|---------|------------|
| CrossNER | 53.2 | 47.1 | 52.1 |
| MIT | 43.2 | 37.4 | 41.5 |
| Average | 48.2 | 42.3 | 46.8 |

*Note: GLiNER is 140x smaller than UniNER-13B*

---

## ✅ Lesson Summary

In this lesson, we covered:

1. **Zero-Shot NER**: Extract any entity type without training
2. **GLiNER Architecture**: How entity prompts enable flexible NER
3. **Custom Entity Types**: Domain-specific extraction made easy
4. **Configuration**: Threshold tuning and nested NER
5. **Comparison**: When to use GLiNER vs traditional NER
6. **Best Practices**: Label engineering and production tips

### 🚀 Next Lesson Preview

In **Lesson 5**, we'll learn **Fine-tuning NER Models**, including:
- Preparing custom NER datasets
- Fine-tuning BERT for specific domains
- Evaluation and hyperparameter tuning

In [ ]:
print("🎉 Congratulations! You've completed Lesson 4: GLiNER - Zero-Shot NER")
print("\n📝 Key takeaways:")
print("   1. GLiNER enables NER for any entity type without training")
print("   2. Labels should be lowercase or title case for best results")
print("   3. Threshold controls precision/recall trade-off")
print("   4. GLiNER is ideal for rapid prototyping and custom domains")
print("   5. Use specific, focused labels for better accuracy")
print("\n👉 Continue to Lesson 5: Fine-tuning NER Models")